# Componentes fuertemente conexas

En un grafo dirigido, una componente fuertemente conexa (CFC) es un conjunto máximo de nodos en el que todos se alcanzan entre sí. Kosaraju hace un DFS para obtener el orden de finalización y otro sobre el grafo traspuesto en orden inverso; Tarjan lo hace en un solo DFS con los valores `indice` y `bajo`. Ambos cuestan $O(V + E)$. Al contraer cada CFC en un nodo queda un grafo acíclico (el grafo condensado).

## 1. Kosaraju

Implementar Kosaraju y aplicarlo a una red de cuentas que se siguen en una red social.

In [1]:
import random
import sys

sys.setrecursionlimit(10_000)

def kosaraju(grafo):
    visitado, orden = set(), []

    def dfs(u):
        visitado.add(u)
        for v in grafo.get(u, ()):
            if v not in visitado:
                dfs(v)
        orden.append(u)

    for u in grafo:
        if u not in visitado:
            dfs(u)
    traspuesto = {u: [] for u in grafo}
    for u in grafo:
        for v in grafo[u]:
            traspuesto.setdefault(v, []).append(u)
    componentes, asignado = [], set()
    for u in reversed(orden):
        if u in asignado:
            continue
        pila, comp = [u], []
        asignado.add(u)
        while pila:
            x = pila.pop()
            comp.append(x)
            for y in traspuesto.get(x, ()):
                if y not in asignado:
                    asignado.add(y)
                    pila.append(y)
        componentes.append(sorted(comp))
    return componentes

sigue = {"ana": ["luis"], "luis": ["eva"], "eva": ["ana", "pau"], "pau": ["noa"], "noa": ["pau", "iker"], "iker": []}
print(kosaraju(sigue))

[['ana', 'eva', 'luis'], ['noa', 'pau'], ['iker']]


## 2. Tarjan

Implementar Tarjan con un solo DFS y comprobar en grafos aleatorios que da las mismas componentes que Kosaraju y que la definición (dos nodos están en la misma CFC si cada uno alcanza al otro).

In [2]:
def tarjan(grafo):
    indice, bajo, en_pila, pila, componentes = {}, {}, set(), [], []

    def dfs(u):
        indice[u] = bajo[u] = len(indice)
        pila.append(u)
        en_pila.add(u)
        for v in grafo.get(u, ()):
            if v not in indice:
                dfs(v)
                bajo[u] = min(bajo[u], bajo[v])
            elif v in en_pila:
                bajo[u] = min(bajo[u], indice[v])
        if bajo[u] == indice[u]:
            comp = []
            while True:
                x = pila.pop()
                en_pila.discard(x)
                comp.append(x)
                if x == u:
                    break
            componentes.append(sorted(comp))

    for u in grafo:
        if u not in indice:
            dfs(u)
    return componentes

def alcanzables(grafo, u):
    vistos, pila = {u}, [u]
    while pila:
        for v in grafo.get(pila.pop(), ()):
            if v not in vistos:
                vistos.add(v)
                pila.append(v)
    return vistos

random.seed(1)
for _ in range(300):
    n = random.randint(1, 10)
    g = {u: [v for v in range(n) if v != u and random.random() < 0.2] for u in range(n)}
    a, b = sorted(kosaraju(g)), sorted(tarjan(g))
    assert a == b
    alc = {u: alcanzables(g, u) for u in g}
    for comp in a:
        for u in comp:
            assert set(comp) == {v for v in g if v in alc[u] and u in alc[v]}
print(tarjan(sigue))

[['iker'], ['noa', 'pau'], ['ana', 'eva', 'luis']]


## 3. Grafo condensado

Contraer cada CFC del ejemplo de la red social en un nodo, construir el grafo condensado y comprobar que es acíclico obteniendo un orden topológico.

In [3]:
from graphlib import TopologicalSorter

def condensado(grafo):
    comps = tarjan(grafo)
    de = {u: i for i, comp in enumerate(comps) for u in comp}
    aristas = {i: set() for i in range(len(comps))}
    for u in grafo:
        for v in grafo[u]:
            if de[u] != de[v]:
                aristas[de[u]].add(de[v])
    return comps, aristas

comps, aristas = condensado(sigue)
for i, comp in enumerate(comps):
    print(i, comp, "->", sorted(aristas[i]))
# TopologicalSorter espera predecesores: se le pasa el grafo traspuesto
predecesores = {i: {j for j in aristas if i in aristas[j]} for i in aristas}
print("orden:", list(TopologicalSorter(predecesores).static_order()))

0 ['iker'] -> []
1 ['noa', 'pau'] -> [0]
2 ['ana', 'eva', 'luis'] -> [1]
orden: [2, 1, 0]


## 4. 2-SAT

Resolver un problema 2-SAT: cada cláusula es $(a \lor b)$ con literales que pueden estar negados. Se construye el grafo de implicaciones ($\lnot a \Rightarrow b$, $\lnot b \Rightarrow a$) y la fórmula es satisfacible si ninguna variable está en la misma CFC que su negación. Aplicarlo a decidir qué charlas de una jornada van por la mañana con restricciones de pares y comprobarlo contra fuerza bruta.

In [4]:
from itertools import product

def dos_sat(n, clausulas):
    # literal: (variable, valor); nodo 2*i es x_i verdadero, 2*i+1 es x_i falso
    nodo = lambda var, val: 2 * var + (0 if val else 1)
    grafo = {i: [] for i in range(2 * n)}
    for (a, va), (b, vb) in clausulas:
        grafo[nodo(a, not va)].append(nodo(b, vb))
        grafo[nodo(b, not vb)].append(nodo(a, va))
    comps = tarjan(grafo)  # Tarjan las devuelve en orden topológico inverso
    de = {u: i for i, comp in enumerate(comps) for u in comp}
    if any(de[2 * i] == de[2 * i + 1] for i in range(n)):
        return None
    return [de[2 * i] < de[2 * i + 1] for i in range(n)]

# x_i = True: la charla i va por la mañana
clausulas = [((0, True), (1, True)), ((0, False), (2, False)), ((1, False), (2, True)), ((2, True), (3, True))]
print(dos_sat(4, clausulas))

cumple = lambda asig, cl: all(asig[a] == va or asig[b] == vb for (a, va), (b, vb) in cl)
random.seed(2)
for _ in range(500):
    n = random.randint(1, 5)
    cl = [((random.randrange(n), random.random() < 0.5), (random.randrange(n), random.random() < 0.5)) for _ in range(random.randint(1, 8))]
    sol = dos_sat(n, cl)
    existe = any(cumple(asig, cl) for asig in product((False, True), repeat=n))
    assert (sol is not None) == existe
    if sol:
        assert cumple(sol, cl)

[True, False, False, True]
